<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]

In [ ]:
!pip install --upgrade torchao

In [ ]:
!pip install trl[GRPOTrainer]

In [ ]:
!pip install trl[vllm]

  Using cached vllm-0.25.1-cp38-abi3-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached vllm-0.25.1-cp38-abi3-manylinux_2_28_x86_64.whl (250.1 MB)


In [ ]:
!pip install trl[GRPOTrainer]

In [ ]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
import re

In [ ]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
import re

In [ ]:
print(train_dataset[0].keys())


dict_keys(['category', 'difficulty', 'quality', 'reward_model_score', 'conversation_tokens', 'prompt', 'completion'])


In [ ]:
import re
from datasets import load_dataset

# 1. Dataset Preparation
train_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:200]")
eval_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:200]")
model_name = "HuggingFaceTB/SmolLM2-135M"

def extract_prompt_completion_from_messages(item):
    prompt = None
    completion = None
    for message in item['messages']:
        if message['role'] == 'user':
            prompt = message['content']
        elif message['role'] == 'assistant':
            completion = message['content']
    return {'prompt': prompt, 'completion': completion}

prompt_to_gold_completion_map = {}
for item in train_dataset:
    extracted = extract_prompt_completion_from_messages(item)
    if extracted['prompt'] is not None and extracted['completion'] is not None:
        prompt_to_gold_completion_map[extracted['prompt']] = extracted['completion']

train_dataset = train_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])
eval_dataset = eval_dataset.map(extract_prompt_completion_from_messages, remove_columns=['messages'])

train_dataset = train_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)
eval_dataset = eval_dataset.filter(lambda x: x['prompt'] is not None and x['completion'] is not None)


# 2. INSERTED: Format Reward Function
def reward_format(completions, **kwargs):
    """
    Rewards completions that contain a proper XML structure,
    such as <think>...</think> and <answer>...</answer> tags.
    """
    rewards = []
    # Pattern to check for matching open and close tags
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"

    for completion in completions:
        # Clean up whitespace and check against the pattern
        clean_completion = completion.strip()
        if re.match(pattern, clean_completion, re.DOTALL):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


# 3. Ground Truth Reward Function
def ground_truth_check_reward(prompts, completions, **kwargs):
    rewards = []
    reward_fn_kwargs = kwargs.get('reward_fn_kwargs', {})
    prompt_to_gold_completion_map = reward_fn_kwargs.get('prompt_to_gold_completion_map', {})

    for prompt, completion in zip(prompts, completions):
        gold_completion = prompt_to_gold_completion_map.get(prompt, None)
        if gold_completion and completion == gold_completion:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


# 4. Combined Reward Function
def combined_reward_function(prompts, completions, **kwargs):
    format_rewards = reward_format(completions, **kwargs)
    truth_rewards = ground_truth_check_reward(prompts, completions, **kwargs)

    combined = [ (f_r + t_r) / 2.0 for f_r, t_r in zip(format_rewards, truth_rewards) ]
    return combined


In [ ]:
# training config
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    logging_steps=10,
)

In [ ]:
# more detailed config (from the website)
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    num_generations=4,  # Number of completions to generate for each prompt
    per_device_train_batch_size=4,  # We want to get all generations in one device batch
    # Optional but useful
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    # GRPO specific (optional)
    use_vllm=False  # Speed up generation, need to 'pip install trl[vllm]'. Disabled due to CUDA library error.
)

In [ ]:
# train_grpo.py
trainer = GRPOTrainer(
    model=model_name,
    args=training_args,
    reward_funcs=combined_reward_function,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
    # reward_kwargs={'reward_args': {'prompt_to_gold_completion_map': prompt_to_gold_completion_map}}
)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000


Step,Training Loss
10,0.000000
20,0.000000
30,0.000000
40,0.000000
50,0.000000
60,0.000000
70,0.000000
80,0.000000
90,0.000000
100,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=300, training_loss=0.0, metrics={'train_runtime': 5025.6719, 'train_samples_per_second': 0.119, 'train_steps_per_second': 0.06, 'total_flos': 0.0, 'train_loss': 0.0, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/combined Reward Function/mean,Rewards/combined Reward Function/std,Reward,Reward Std,Frac Reward Zero Std,Entropy,Clip Ratio/low Mean,Clip Ratio/high Mean,Clip Ratio/region Mean,Clip Ratio/low Min,Clip Ratio/high Max
No log,0.000000,0,0.000000,229.892500,119.240000,256.000000,0.815000,85.246667,62.920000,108.100000,0.000000,0.000000,0.000000,0.000000,1.000000,3.417125,0.000000,0.000000,0.000000,0.000000,0.000000


{'eval_loss': 0.0,
 'eval_num_tokens': 0.0,
 'eval_completions/mean_length': 229.8925,
 'eval_completions/min_length': 119.24,
 'eval_completions/max_length': 256.0,
 'eval_completions/clipped_ratio': 0.815,
 'eval_completions/mean_terminated_length': 85.2466671514511,
 'eval_completions/min_terminated_length': 62.92,
 'eval_completions/max_terminated_length': 108.1,
 'eval_rewards/combined_reward_function/mean': 0.0,
 'eval_rewards/combined_reward_function/std': 0.0,
 'eval_reward': 0.0,
 'eval_reward_std': 0.0,
 'eval_frac_reward_zero_std': 1.0,
 'eval_entropy': 3.417125198841095,
 'eval_clip_ratio/low_mean': 0.0,
 'eval_clip_ratio/high_mean': 0.0,
 'eval_clip_ratio/region_mean': 0.0,
 'eval_clip_ratio/low_min': 0.0,
 'eval_clip_ratio/high_max': 0.0}